In [1]:
import pandas as pd
import tqdm as tqdm

In [2]:
from huggingface_hub import upload_folder

## Data analysis

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Huggingface_Api_Key")
from huggingface_hub import login

login(token=secret_value_0)

In [4]:
from huggingface_hub import HfApi
from huggingface_hub import HfApi

# api = HfApi()

# repo_id = "YOUR_HUGGINGFACE_USERNAME/kenyan-media-bias-qwen"

# api.create_repo(
#     repo_id=repo_id,
#     private=True,
#     exist_ok=True
# )

# print("Repository:", repo_id)

In [5]:
import torch

print("GPU count:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))

GPU count: 2
GPU: Tesla T4


In [6]:
# !pip install -q transformers datasets accelerate sentencepiece evaluate rouge_score

In [7]:
import pandas as pd
import numpy as np
import torch
import json
import ast

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

In [8]:
import pandas as pd

df = pd.read_parquet('/kaggle/input/datasets/rikufootballcorner/political-verified/political_verified.parquet')

df.columns

Index(['url', 'website', 'article_text', 'political_classification',
       'bias_classification', 'bias_reason', 'judge_evidence_present',
       'judge_evidence_type', 'judge_evidence_quote', 'judge_verifier_reason',
       'judge_independent_label', 'judge_initial_label_supported',
       'judge_initial_reason_supported', 'judge_disagreement_reason',
       'judge_confidence', 'luna_label_norm', 'agree', 'quote_grounded'],
      dtype='object')

In [9]:
print(df.columns)
print(df.shape)

Index(['url', 'website', 'article_text', 'political_classification',
       'bias_classification', 'bias_reason', 'judge_evidence_present',
       'judge_evidence_type', 'judge_evidence_quote', 'judge_verifier_reason',
       'judge_independent_label', 'judge_initial_label_supported',
       'judge_initial_reason_supported', 'judge_disagreement_reason',
       'judge_confidence', 'luna_label_norm', 'agree', 'quote_grounded'],
      dtype='object')
(16367, 18)


In [10]:
df.drop_duplicates(subset=['url'],inplace=True)

In [11]:
df_agree=df[df['agree']==True]

In [12]:
df_agree['bias_classification'].value_counts()

bias_classification
not_biased    10473
biased         1163
Name: count, dtype: int64

In [13]:
biased_df = df_agree[
    df_agree["bias_classification"] == "biased"
]

not_biased_df = df_agree[
    df_agree["bias_classification"] == "not_biased"
].sample(
    n=len(biased_df) * 3,
    random_state=42
)

df_3to1 = pd.concat(
    [biased_df, not_biased_df],
    ignore_index=True
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [14]:
df_3to1.to_csv('final_sample1.csv', index=False)

In [15]:
from sklearn.model_selection import train_test_split

# First split: 80% train, 20% temporary
train_df, temp_df = train_test_split(
    df_3to1,
    test_size=0.20,
    random_state=42,
    stratify=df_3to1["bias_classification"]
)

# Second split: 10% validation, 10% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["bias_classification"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain distribution:")
print(train_df["bias_classification"].value_counts())

print("\nValidation distribution:")
print(val_df["bias_classification"].value_counts())

print("\nTest distribution:")
print(test_df["bias_classification"].value_counts())

Train: 3721
Validation: 465
Test: 466

Train distribution:
bias_classification
not_biased    2791
biased         930
Name: count, dtype: int64

Validation distribution:
bias_classification
not_biased    349
biased        116
Name: count, dtype: int64

Test distribution:
bias_classification
not_biased    349
biased        117
Name: count, dtype: int64


In [16]:
PROMPT_TEMPLATE = """
You are an expert annotator evaluating framing bias in Kenyan political news articles.

TASK
Determine whether the article contains textual evidence of biased framing that favors or disadvantages a political actor, political side, or political perspective.

Evaluate the article's presentation, not whether its political claims are true or whether you agree with them.

ASSESSMENT CRITERIA

Consider the following indicators:
1. Loaded language: emotionally charged or evaluative wording that portrays an actor favorably or unfavorably beyond straightforward reporting.
2. One-sided framing: presenting one political side's interpretation while excluding relevant competing perspectives.
3. Unequal treatment: describing, scrutinizing, or contextualizing comparable political actors differently.
4. Selective emphasis: emphasizing particular facts, allegations, or events in a way that creates a discernible political slant.
5. Perspective omission: excluding a relevant perspective when the omission materially affects the article's framing.

EVIDENCE RULES

- Base your decision exclusively on the supplied article.
- Identify specific words, phrases, descriptions, quotations, or presentation choices that support your decision.
- Distinguish the journalist's framing from claims or opinions attributed to politicians, witnesses, or other sources.
- Do not assume that an article is biased because of the politician, party, institution, topic, or news organization involved.
- Political criticism, allegations, corruption reports, government failures, and opposition claims are not automatically evidence of media bias.
- Do not assume that every article must give equal space to every political side.
- Do not invent quotations, omitted perspectives, or contextual facts.
- If there is insufficient textual evidence of biased framing, classify the article as not_biased.

CLASSIFICATION

Use exactly one label:
- biased: identifiable textual evidence indicates that the article's framing favors or disadvantages a political actor, side, or perspective.
- not_biased: the article is predominantly factual or balanced, or the available text does not provide sufficient evidence of biased framing.

OUTPUT FORMAT

Return exactly these two fields:

bias_classification: biased OR not_biased
bias_reason: A concise explanation grounded in specific evidence from the article. For biased articles, identify the evidence and explain its framing effect. For not_biased articles, explain the relevant features supporting the decision or state that sufficient evidence of biased framing was not found.

Do not add headings, scores, recommendations, or additional fields.
"""

def make_text(row):
    return f"""{PROMPT_TEMPLATE}

ARTICLE:
{row["article_text"]}

ANSWER:
bias_classification: {row["bias_classification"]}
bias_reason: {row["bias_reason"]}
"""

In [17]:
import torch
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

In [18]:
train_df = train_df.copy()
val_df = val_df.copy()

train_df["text"] = train_df.apply(make_text, axis=1)
val_df["text"] = val_df.apply(make_text, axis=1)
test_df["text"] = test_df.apply(make_text, axis=1)

In [19]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["text"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["text"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text"]],
    preserve_index=False
)

In [20]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [21]:
# # model_name = "gpt2"

# # tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# # tokenizer.pad_token = tokenizer.eos_token

# # model = GPT2LMHeadModel.from_pretrained(model_name)

# # model.config.pad_token_id = tokenizer.pad_token_id


# MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True
# )

# tokenizer = AutoTokenizer.from_pretrained(
#     MODEL_NAME,
#     trust_remote_code=True
# )

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True
# )

# model.config.use_cache = False

# print("Model loaded.")

In [22]:
# model = prepare_model_for_kbit_training(model)

# lora_config = LoraConfig(
#     r=8,
#     lora_alpha=16,
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM",
#     target_modules=[
#         "q_proj",
#         "k_proj",
#         "v_proj",
#         "o_proj"
#     ]
# )

# model = get_peft_model(model, lora_config)

# model.print_trainable_parameters()

In [23]:
!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 50.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Load your fine-tuned model from Hugging Face
model_name = "Mriku/kenyan-media-bias-qwen"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


adapter_model.safetensors:   0%|          | 0.00/14.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Model loaded successfully!


In [25]:
MAX_LENGTH = 1024

In [26]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

In [27]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/3721 [00:00<?, ? examples/s]

Map:   0%|          | 0/465 [00:00<?, ? examples/s]

In [28]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [29]:
# training_args = TrainingArguments(
#     output_dir="/kaggle/working/qwen",

#     num_train_epochs=3,

#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=2,

#     gradient_accumulation_steps=4,

#     learning_rate=5e-5,

#     weight_decay=0.01,

#     eval_strategy="epoch",
#     save_strategy="epoch",

#     logging_steps=50,

#     load_best_model_at_end=True,

#     report_to="none",

#     fp16=torch.cuda.is_available()
# )

In [30]:
# trainer = Trainer(
#     model=model,
#     args=training_args,

#     train_dataset=tokenized_train,
#     eval_dataset=tokenized_val,

#     processing_class=tokenizer,

#     data_collator=data_collator
# )

In [31]:
# trainer.train()

In [32]:
# model.eval()


In [33]:
def predict_article(article):

    prompt = f"""{PROMPT_TEMPLATE}

ARTICLE:
{article}

ANSWER:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Remove the prompt from the generated output
    generated_tokens = output[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [34]:
article = test_df["article_text"].iloc[2]

prediction = predict_article(article)

print(prediction)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


bias_classification: not_biased
bias_reason: The article presents the incident through multiple accounts, including statements from hotel proprietor Otieno Odongo and a senior political leader, without clearly favoring or disparaging either side. It reports the youths' alleged actions, damage to property, and the subsequent blocking of the road, but does not use loaded language, present a single political interpretation, or omit a relevant perspective. The claims about the Linda Mwananchi movement and the one-term slogan are attributed to the political leader rather than established as established fact.
Loading…





TASK
Determine whether the article contains textual evidence of biased framing that favors or disadvantages a political actor, political side, or political perspective.

Evaluate the article's presentation, not whether its political claims are true or whether you agree with them.

ASSESSMENT CRITERIA

Consider the following indicators:
1. Loaded language: emotionally charg

In [35]:
test_df.iloc[1]

url                               https://www.the-star.co.ke/counties/nyanza/202...
website                                                                        Star
article_text                      Kisumu Governor Anyang Nyong’o/ Faith Matete\n...
political_classification                                                  Political
bias_classification                                                      not_biased
bias_reason                       The article presents the county government's d...
judge_evidence_present                                                        False
judge_evidence_type                                                              []
judge_evidence_quote                                                               
judge_verifier_reason             The article is a standard report on a statemen...
judge_independent_label                                                  not_biased
judge_initial_label_supported                                               

In [36]:
df.head()

,url,website,article_text,political_classification,bias_classification,bias_reason,judge_evidence_present,judge_evidence_type,judge_evidence_quote,judge_verifier_reason,judge_independent_label,judge_initial_label_supported,judge_initial_reason_supported,judge_disagreement_reason,judge_confidence,luna_label_norm,agree,quote_grounded
0,https://capitalfm.africa/opposition-demands-an...,Capital FM,"NAIROBI, Kenya, Aug 3 – Opposition leaders on ...",Political,not_biased,The article primarily reports the opposition l...,False,[],,"The article is a factual, objective account of...",not_biased,True,True,,high,not_biased,True,None
1,https://capitalfm.africa/mukurweini-mp-kaguchi...,Capital FM,"NAIROBI, Kenya, Aug 3– Mukurweini Member of Pa...",Political,not_biased,"The article uses predominantly factual, attrib...",False,[],,"The article provides an objective, straightfor...",not_biased,True,True,,high,not_biased,True,None
2,https://capitalfm.africa/kuria-revives-bbi-tal...,Capital FM,"NAIROBI,Kenya Aug 3 – Former Cabinet Secretary...",Political,not_biased,The article predominantly uses factual and att...,False,[],,The article is a standard news report covering...,not_biased,True,True,,high,not_biased,True,None
3,https://capitalfm.africa/kindiki-declares-ruto...,Capital FM,"NAIROBI, Kenya Aug 3 – Deputy President Kithur...",Political,biased,The article frames Kindiki favorably by presen...,True,"[loaded_language, selective_framing]",to enable him easily fend off the feeble chall...,The article adopts subjective and dismissive l...,biased,True,True,None. The initial evaluation accurately identi...,high,biased,True,True
4,https://capitalfm.africa/ruto-unveils-measures...,Capital FM,"NAIROBI, Kenya Aug 3 – The Government is imple...",Political,biased,"The article gives extensive, largely unchallen...",False,[],,The article is routine event-driven reporting ...,not_biased,False,False,Luna conflates political rhetoric delivered at...,high,biased,False,None


In [37]:
df.to_csv('cleaned_articles.csv')

In [38]:
df.columns

Index(['url', 'website', 'article_text', 'political_classification',
       'bias_classification', 'bias_reason', 'judge_evidence_present',
       'judge_evidence_type', 'judge_evidence_quote', 'judge_verifier_reason',
       'judge_independent_label', 'judge_initial_label_supported',
       'judge_initial_reason_supported', 'judge_disagreement_reason',
       'judge_confidence', 'luna_label_norm', 'agree', 'quote_grounded'],
      dtype='object')

In [39]:
# clean_output_dir = "/kaggle/working/final_qwen_model"
# trainer.save_model(clean_output_dir)
# tokenizer.save_pretrained(clean_output_dir)
# repo_id = "Mriku/kenyan-media-bias-qwen"


In [40]:
import re
import gc
import torch
import pandas as pd

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from rouge_score import rouge_scorer


# ============================================================
# 2. LOAD YOUR FINE-TUNED HUGGING FACE MODEL
# ============================================================

MODEL_ID = "Mriku/kenyan-media-bias-qwen"

if "model" in globals():
    del model

if "tokenizer" in globals():
    del tokenizer

gc.collect()

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required.")

torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Loaded fine-tuned model:", MODEL_ID)


# ============================================================
# 3. PREDICTION FUNCTION
#    Uses your existing PROMPT_TEMPLATE
# ============================================================

def predict_article(article):

    prompt = f"""{PROMPT_TEMPLATE}

ARTICLE:
{article}

ANSWER:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


# ============================================================
# 4. EXTRACT CLASSIFICATION AND REASON
# ============================================================

def extract_prediction(prediction):

    prediction = str(prediction)

    cls_match = re.search(
        r"(?:bias_classification|bias classification|Label)"
        r"\s*:\s*(not_biased|biased)\b",
        prediction,
        re.IGNORECASE
    )

    if cls_match is None:
        cls_match = re.match(
            r"\s*(not_biased|biased)\b",
            prediction,
            re.IGNORECASE
        )

    reason_match = re.search(
        r"(?:bias_reason|bias reason|Reason)"
        r"\s*:\s*(.*?)"
        r"(?=\n\s*(?:Continue annotating|Label\s*:|"
        r"WoW Newsroom|WoW Newsroom is licensed|"
        r"WoW Newsroom uses articles)|\Z)",
        prediction,
        re.IGNORECASE | re.DOTALL
    )

    predicted_class = (
        cls_match.group(1).lower()
        if cls_match else None
    )

    predicted_reason = (
        reason_match.group(1).strip()
        if reason_match else None
    )

    return predicted_class, predicted_reason


# ============================================================
# 5. PREPARE TEST DATA
# ============================================================

required_columns = [
    "article_text",
    "bias_classification",
    "bias_reason"
]

missing_columns = [
    col for col in required_columns
    if col not in test_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in test_df: {missing_columns}"
    )

evaluation_df = test_df.dropna(
    subset=required_columns
).copy()

evaluation_df["bias_classification"] = (
    evaluation_df["bias_classification"]
    .astype(str)
    .str.strip()
    .str.lower()
)

evaluation_df = evaluation_df[
    evaluation_df["bias_classification"].isin(
        ["biased", "not_biased"]
    )
].reset_index(drop=True)

print("Articles to evaluate:", len(evaluation_df))
print("\nReference class distribution:")
print(evaluation_df["bias_classification"].value_counts())


# ============================================================
# 6. GENERATE PREDICTIONS
# ============================================================

results = []

for _, row in tqdm(
    evaluation_df.iterrows(),
    total=len(evaluation_df),
    desc="Evaluating fine-tuned Qwen"
):
    try:
        raw_prediction = predict_article(
            row["article_text"]
        )

        predicted_class, predicted_reason = (
            extract_prediction(raw_prediction)
        )

        results.append({
            "article_text": row["article_text"],
            "actual_class": row["bias_classification"],
            "predicted_class": predicted_class,
            "reference_reason": str(row["bias_reason"]),
            "predicted_reason": predicted_reason,
            "raw_prediction": raw_prediction
        })

    except Exception as e:
        print(f"\nPrediction error: {e}")

        results.append({
            "article_text": row["article_text"],
            "actual_class": row["bias_classification"],
            "predicted_class": None,
            "reference_reason": str(row["bias_reason"]),
            "predicted_reason": None,
            "raw_prediction": None
        })

results_df = pd.DataFrame(results)


# ============================================================
# 7. CLASSIFICATION METRICS
# ============================================================

valid_cls = results_df[
    results_df["predicted_class"].isin(
        ["biased", "not_biased"]
    )
].copy()

print("\n" + "=" * 55)
print("FINE-TUNED QWEN: CLASSIFICATION RESULTS")
print("=" * 55)

print("Total test articles:", len(results_df))
print("Valid predictions:", len(valid_cls))
print("Unparsed predictions:", len(results_df) - len(valid_cls))

if len(valid_cls) > 0:

    y_true = valid_cls["actual_class"]
    y_pred = valid_cls["predicted_class"]

    accuracy = accuracy_score(y_true, y_pred)

    print(f"\nAccuracy: {accuracy:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=["biased", "not_biased"],
            zero_division=0,
            digits=4
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=["biased", "not_biased"]
    )

    cm_df = pd.DataFrame(
        cm,
        index=["Actual biased", "Actual not_biased"],
        columns=["Predicted biased", "Predicted not_biased"]
    )

    print("\nConfusion matrix:")
    print(cm_df)

    end_to_end_accuracy = (
        (y_true == y_pred).sum() / len(results_df)
        if len(results_df) > 0 else 0
    )

    print(
        "\nEnd-to-end accuracy (unparsed outputs count as incorrect):",
        f"{end_to_end_accuracy:.4f}"
    )

else:
    print("No valid predictions available for classification metrics.")


# ============================================================
# 8. ROUGE SCORES FOR GENERATED REASONS
# ============================================================

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

rouge_results = []

for _, row in results_df.iterrows():

    reference = row["reference_reason"]
    generated = row["predicted_reason"]

    if not isinstance(reference, str):
        continue

    if not isinstance(generated, str):
        continue

    if not reference.strip() or not generated.strip():
        continue

    scores = scorer.score(
        reference.strip(),
        generated.strip()
    )

    rouge_results.append({
        "rouge1_precision": scores["rouge1"].precision,
        "rouge1_recall": scores["rouge1"].recall,
        "rouge1_f1": scores["rouge1"].fmeasure,
        "rouge2_precision": scores["rouge2"].precision,
        "rouge2_recall": scores["rouge2"].recall,
        "rouge2_f1": scores["rouge2"].fmeasure,
        "rougeL_precision": scores["rougeL"].precision,
        "rougeL_recall": scores["rougeL"].recall,
        "rougeL_f1": scores["rougeL"].fmeasure
    })

print("\n" + "=" * 55)
print("FINE-TUNED QWEN: ROUGE RESULTS")
print("=" * 55)

rouge_df = pd.DataFrame(rouge_results)

if not rouge_df.empty:
    print("Reasons evaluated:", len(rouge_df))
    print("\nMean ROUGE scores:")
    print(rouge_df.mean().round(4).to_string())
else:
    print("No valid explanation pairs were available.")


# ============================================================
# 9. SAVE DETAILED RESULTS
# ============================================================

results_df.to_csv(
    "finetuned_qwen_evaluation.csv",
    index=False
)

print("\nSaved detailed predictions:")
print("finetuned_qwen_evaluation.csv")


# ============================================================
# 10. SAVE METRICS SUMMARY
# ============================================================

summary = {
    "model": MODEL_ID,
    "total_articles": len(results_df),
    "valid_predictions": len(valid_cls),
    "unparsed_predictions": len(results_df) - len(valid_cls)
}

if len(valid_cls) > 0:

    report = classification_report(
        valid_cls["actual_class"],
        valid_cls["predicted_class"],
        labels=["biased", "not_biased"],
        output_dict=True,
        zero_division=0
    )

    summary.update({
        "accuracy_valid_predictions": accuracy_score(
            valid_cls["actual_class"],
            valid_cls["predicted_class"]
        ),
        "end_to_end_accuracy": end_to_end_accuracy,
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "biased_precision": report["biased"]["precision"],
        "biased_recall": report["biased"]["recall"],
        "biased_f1": report["biased"]["f1-score"],
        "not_biased_precision": report["not_biased"]["precision"],
        "not_biased_recall": report["not_biased"]["recall"],
        "not_biased_f1": report["not_biased"]["f1-score"]
    })

if not rouge_df.empty:
    for column in rouge_df.columns:
        summary[column] = rouge_df[column].mean()

pd.DataFrame([summary]).to_csv(
    "finetuned_qwen_metrics_summary.csv",
    index=False
)

print("\nSaved metrics summary:")
print("finetuned_qwen_metrics_summary.csv")


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded fine-tuned model: Mriku/kenyan-media-bias-qwen
Articles to evaluate: 466

Reference class distribution:
bias_classification
not_biased    349
biased        117
Name: count, dtype: int64


Evaluating fine-tuned Qwen:   0%|          | 0/466 [00:00<?, ?it/s]


FINE-TUNED QWEN: CLASSIFICATION RESULTS
Total test articles: 466
Valid predictions: 438
Unparsed predictions: 28

Accuracy: 0.7740

Classification report:
              precision    recall  f1-score   support

      biased     1.0000    0.0100    0.0198       100
  not_biased     0.7735    1.0000    0.8723       338

    accuracy                         0.7740       438
   macro avg     0.8867    0.5050    0.4460       438
weighted avg     0.8252    0.7740    0.6776       438


Confusion matrix:
                   Predicted biased  Predicted not_biased
Actual biased                     1                    99
Actual not_biased                 0                   338

End-to-end accuracy (unparsed outputs count as incorrect): 0.7275

FINE-TUNED QWEN: ROUGE RESULTS
Reasons evaluated: 438

Mean ROUGE scores:
rouge1_precision    0.2813
rouge1_recall       0.5479
rouge1_f1           0.3690
rouge2_precision    0.0782
rouge2_recall       0.1550
rouge2_f1           0.1032
rougeL_precision    

In [41]:

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import re
import gc
import torch
import pandas as pd

from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from rouge_score import rouge_scorer


# ============================================================
# 2. LOAD THE ORIGINAL QWEN MODEL
# ============================================================

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Free GPU memory from the previously loaded model, if applicable
if "model" in globals():
    del model

if "tokenizer" in globals():
    del tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
else:
    raise RuntimeError("A CUDA GPU is required for this code.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto"
)

model.eval()

print("Loaded original model:", MODEL_ID)


# ============================================================
# 3. PREDICTION FUNCTION
#    Uses your existing PROMPT_TEMPLATE unchanged
# ============================================================

def predict_article(article):

    prompt = f"""{PROMPT_TEMPLATE}

ARTICLE:
{article}

ANSWER:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    # Place inputs on the model's input device
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


# ============================================================
# 4. EXTRACT CLASSIFICATION AND REASON
# ============================================================

def extract_prediction(prediction):

    prediction = str(prediction)

    # Accept the field names used in your annotation format
    cls_match = re.search(
        r"(?:bias_classification|bias classification|Label)"
        r"\s*:\s*(biased|not_biased)\b",
        prediction,
        re.IGNORECASE
    )

    # Also handle a response beginning directly with the label
    if cls_match is None:
        cls_match = re.match(
            r"\s*(biased|not_biased)\b",
            prediction,
            re.IGNORECASE
        )

    reason_match = re.search(
        r"(?:bias_reason|bias reason|Reason)"
        r"\s*:\s*(.*?)"
        r"(?=\n\s*(?:Continue annotating|Label\s*:|"
        r"WoW Newsroom|WoW Newsroom is licensed|"
        r"WoW Newsroom uses articles)|\Z)",
        prediction,
        re.IGNORECASE | re.DOTALL
    )

    predicted_class = (
        cls_match.group(1).lower()
        if cls_match else None
    )

    predicted_reason = (
        reason_match.group(1).strip()
        if reason_match else None
    )

    return predicted_class, predicted_reason


# ============================================================
# 5. PREPARE THE TEST DATA
# ============================================================

required_columns = [
    "article_text",
    "bias_classification",
    "bias_reason"
]

missing_columns = [
    col for col in required_columns
    if col not in test_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in test_df: {missing_columns}"
    )

evaluation_df = test_df.dropna(
    subset=required_columns
).copy()

evaluation_df["bias_classification"] = (
    evaluation_df["bias_classification"]
    .astype(str)
    .str.strip()
    .str.lower()
)

evaluation_df = evaluation_df[
    evaluation_df["bias_classification"].isin(
        ["biased", "not_biased"]
    )
].reset_index(drop=True)

print("\nArticles to evaluate:", len(evaluation_df))

print("\nReference class distribution:")
print(
    evaluation_df["bias_classification"]
    .value_counts()
)


# ============================================================
# 6. GENERATE BASELINE PREDICTIONS
# ============================================================

results = []

for _, row in tqdm(
    evaluation_df.iterrows(),
    total=len(evaluation_df),
    desc="Evaluating original Qwen"
):

    try:
        raw_prediction = predict_article(
            row["article_text"]
        )

        predicted_class, predicted_reason = (
            extract_prediction(raw_prediction)
        )

        results.append({
            "article_text": row["article_text"],
            "actual_class": row["bias_classification"],
            "predicted_class": predicted_class,
            "reference_reason": str(row["bias_reason"]),
            "predicted_reason": predicted_reason,
            "raw_prediction": raw_prediction
        })

    except Exception as e:
        print(f"\nPrediction error: {e}")

        results.append({
            "article_text": row["article_text"],
            "actual_class": row["bias_classification"],
            "predicted_class": None,
            "reference_reason": str(row["bias_reason"]),
            "predicted_reason": None,
            "raw_prediction": None
        })

results_df = pd.DataFrame(results)


# ============================================================
# 7. CLASSIFICATION METRICS
# ============================================================

valid_cls = results_df[
    results_df["predicted_class"].isin(
        ["biased", "not_biased"]
    )
].copy()

print("\n" + "=" * 55)
print("ORIGINAL QWEN: CLASSIFICATION RESULTS")
print("=" * 55)

print("Total test articles:", len(results_df))
print("Valid predictions:", len(valid_cls))
print(
    "Unparsed predictions:",
    len(results_df) - len(valid_cls)
)

if len(valid_cls) > 0:

    y_true = valid_cls["actual_class"]
    y_pred = valid_cls["predicted_class"]

    accuracy = accuracy_score(y_true, y_pred)

    print(f"\nAccuracy: {accuracy:.4f}")

    print("\nClassification report:")

    print(
        classification_report(
            y_true,
            y_pred,
            labels=["biased", "not_biased"],
            zero_division=0,
            digits=4
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=["biased", "not_biased"]
    )

    cm_df = pd.DataFrame(
        cm,
        index=["Actual biased", "Actual not_biased"],
        columns=[
            "Predicted biased",
            "Predicted not_biased"
        ]
    )

    print("Confusion matrix:")
    print(cm_df)

    print(
        "\nEnd-to-end accuracy, counting unparsed outputs "
        "as unsuccessful:",
        f"{(accuracy_score(y_true, y_pred) * len(valid_cls) / len(results_df)):.4f}"
        if len(results_df) > 0 else "N/A"
    )


# ============================================================
# 8. ROUGE SCORES FOR GENERATED REASONS
# ============================================================

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

rouge_results = []

for _, row in results_df.iterrows():

    reference = row["reference_reason"]
    generated = row["predicted_reason"]

    if not isinstance(reference, str):
        continue

    if not isinstance(generated, str):
        continue

    if not reference.strip() or not generated.strip():
        continue

    scores = scorer.score(
        reference.strip(),
        generated.strip()
    )

    rouge_results.append({
        "rouge1_precision": scores["rouge1"].precision,
        "rouge1_recall": scores["rouge1"].recall,
        "rouge1_f1": scores["rouge1"].fmeasure,
        "rouge2_precision": scores["rouge2"].precision,
        "rouge2_recall": scores["rouge2"].recall,
        "rouge2_f1": scores["rouge2"].fmeasure,
        "rougeL_precision": scores["rougeL"].precision,
        "rougeL_recall": scores["rougeL"].recall,
        "rougeL_f1": scores["rougeL"].fmeasure
    })

print("\n" + "=" * 55)
print("ORIGINAL QWEN: ROUGE RESULTS")
print("=" * 55)

if rouge_results:

    rouge_df = pd.DataFrame(rouge_results)

    print("Reasons evaluated:", len(rouge_df))
    print("\nMean ROUGE scores:")
    print(rouge_df.mean().round(4).to_string())

else:
    print("No valid explanation pairs were available.")


# ============================================================
# 9. SAVE DETAILED RESULTS
# ============================================================

results_df.to_csv(
    "default_qwen_evaluation.csv",
    index=False
)

print("\nSaved detailed predictions to:")
print("default_qwen_evaluation.csv")


# Save a compact metrics summary as well
if len(valid_cls) > 0:

    report = classification_report(
        valid_cls["actual_class"],
        valid_cls["predicted_class"],
        labels=["biased", "not_biased"],
        output_dict=True,
        zero_division=0
    )

    summary = {
        "model": MODEL_ID,
        "total_articles": len(results_df),
        "valid_predictions": len(valid_cls),
        "unparsed_predictions": len(results_df) - len(valid_cls),
        "accuracy_valid_predictions": accuracy_score(
            valid_cls["actual_class"],
            valid_cls["predicted_class"]
        ),
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"]
    }

    if rouge_results:
        for column in rouge_df.columns:
            summary[column] = rouge_df[column].mean()

    pd.DataFrame([summary]).to_csv(
        "default_qwen_metrics_summary.csv",
        index=False
    )

    print("Saved metrics summary to:")
    print("default_qwen_metrics_summary.csv")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded original model: Qwen/Qwen2.5-3B-Instruct

Articles to evaluate: 466

Reference class distribution:
bias_classification
not_biased    349
biased        117
Name: count, dtype: int64


Evaluating original Qwen:   0%|          | 0/466 [00:00<?, ?it/s]


ORIGINAL QWEN: CLASSIFICATION RESULTS
Total test articles: 466
Valid predictions: 440
Unparsed predictions: 26

Accuracy: 0.6341

Classification report:
              precision    recall  f1-score   support

      biased     0.3000    0.4455    0.3586       101
  not_biased     0.8069    0.6903    0.7440       339

    accuracy                         0.6341       440
   macro avg     0.5534    0.5679    0.5513       440
weighted avg     0.6905    0.6341    0.6556       440

Confusion matrix:
                   Predicted biased  Predicted not_biased
Actual biased                    45                    56
Actual not_biased               105                   234

End-to-end accuracy, counting unparsed outputs as unsuccessful: 0.5987

ORIGINAL QWEN: ROUGE RESULTS
Reasons evaluated: 440

Mean ROUGE scores:
rouge1_precision    0.2307
rouge1_recall       0.4690
rouge1_f1           0.3059
rouge2_precision    0.0535
rouge2_recall       0.1098
rouge2_f1           0.0712
rougeL_precision    